In [ ]:
from pathlib import Path
import shutil
import pandas as pd

import teehr
from teehr import RemoteReadWriteEvaluation
from teehr.evaluation.spark_session_utils import create_spark_session

import utils.nrcs_utils as nrcs_utils

### Setup eval

In [ ]:
spark = create_spark_session()

In [ ]:
# define evaluation
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

### Setup fetching params

In [ ]:
# get locations list
df = ev.locations.filter("id LIKE 'nrcs%'").to_sdf()
primary_locs = [row["id"] for row in df.select("id").collect()]

# get POR
start_date = "2025-10-01"
end_date = "2026-09-30"

### Query the NRCS observed data and forecast data

In [ ]:
# fetch observed
observations_df = nrcs_utils.fetch_nrcs_observations(
    location_ids=primary_locs,
    start_date=start_date, 
    end_date=end_date, 
)

In [ ]:
# fetch forecast
forecast_df = nrcs_utils.fetch_nrcs_forecasts(
    location_ids=primary_locs,
    start_date=start_date, 
    end_date=end_date, 
)

### Write the timeseries data to the warehouse

In [ ]:
ev.primary_timeseries.load_dataframe(df=observations_df)

In [ ]:
ev.secondary_timeseries.load_dataframe(df=forecast_df)

### Kill spark

In [ ]:
spark.stop()